# Coding race and same-race couples

Respondents and partners are grouped into the same four categories (White, Black, Hispanic, Other)
so they can be compared directly. Following standard US practice, anyone of Hispanic origin is
coded as Hispanic regardless of race, so Black Hispanics are coded as Hispanic.

## 1. Loading the cleaned data

Load the 2,862 partnered respondents saved by `01_clean`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/hcmst2017_clean.csv", low_memory=False)
df.shape

(2862, 730)

## 2. Grouping respondent race

Respondent race/ethnicity (`w1_ppethm`) is reported in five groups. Hispanic respondents of any race
are already coded as Hispanic. We combine "Other, Non-Hispanic" and "2+ Races, Non-Hispanic" into a
single Other group, giving four groups: White, Black, Hispanic, Other.

In [2]:
race_map = {
    "White, Non-Hispanic": "White",
    "Black, Non-Hispanic": "Black",
    "Hispanic": "Hispanic",
    "Other, Non-Hispanic": "Other",
    "2+ Races, Non-Hispanic": "Other",
}
df["race_respondent"] = df["w1_ppethm"].map(race_map)

print(df["w1_ppethm"].value_counts())
print()
print(df["race_respondent"].value_counts(dropna=False))

w1_ppethm
White, Non-Hispanic       2082
Hispanic                   347
Black, Non-Hispanic        229
Other, Non-Hispanic        118
2+ Races, Non-Hispanic      86
Name: count, dtype: int64

race_respondent
White       2082
Hispanic     347
Black        229
Other        204
Name: count, dtype: int64


/var/folders/tk/j75m2rk91ll9tl_rymw764qh0000gn/T/ipykernel_22908/902446410.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["race_respondent"] = df["w1_ppethm"].map(race_map)


## 3. Grouping partner race, with Hispanic origin

Partner race is built from two questions, using the same rule as for respondents:
1. If the partner is Latino or Hispanic (`w1_q6a`, any "Yes" answer) → **Hispanic**, regardless of race.
2. Otherwise, use the partner's race (`w1_q6b`): White → **White**; Black or African American →
   **Black**; Asian or Pacific Islander, American Indian, or other → **Other**.
3. "Refused" or blank → missing.

This fixes the earlier coding, where partner race ignored Hispanic origin and so could not be
matched to respondent race.

In [3]:
def partner_race(row):
    hispanic = str(row["w1_q6a"])
    race = str(row["w1_q6b"])

    # Step 1: Hispanic origin first
    if hispanic.startswith("Yes"):
        return "Hispanic"

    # Step 2: otherwise use race
    if race == "White":
        return "White"
    if race == "Black or African American":
        return "Black"
    if race in ["Asian or Pacific Islander", "American Indian, Aleut, or Eskimo", "Other (please specify)"]:
        return "Other"

    # Step 3: refused or blank
    return None


df["race_partner"] = df.apply(partner_race, axis=1)

df["race_partner"].value_counts(dropna=False)

/var/folders/tk/j75m2rk91ll9tl_rymw764qh0000gn/T/ipykernel_22908/2294864490.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["race_partner"] = df.apply(partner_race, axis=1)


race_partner
White       2092
Hispanic     308
Black        231
Other        217
NaN           14
Name: count, dtype: int64

## 4. Flagging same-race couples

`same_race` is True when the respondent and partner are in the same race group (intraracial couples),
False for interracial couples, and missing when the partner's race is unknown. Our main analysis uses
intraracial couples; interracial couples are kept in the data for the all-couples robustness check.

Note: "Other–Other" couples are not necessarily the same race (e.g. an Asian respondent with a
Native American partner), so our race comparisons focus on White, Black and Hispanic couples.

In [4]:
# Same race? True / False, or missing if partner race is unknown
df["same_race"] = (df["race_respondent"] == df["race_partner"]).astype("boolean")
df.loc[df["race_partner"].isna(), "same_race"] = pd.NA
print(df["same_race"].value_counts(dropna=False))

# Couple types: respondent race (rows) x partner race (columns)
order = ["White", "Black", "Hispanic", "Other"]
couple_types = pd.crosstab(df["race_respondent"], df["race_partner"], margins=True)
couple_types = couple_types.reindex(index=order + ["All"], columns=order + ["All"])
couple_types.to_csv("../output/tables/couple_types_by_race.csv")
couple_types

same_race
True     2280
False     568
<NA>       14
Name: count, dtype: Int64


/var/folders/tk/j75m2rk91ll9tl_rymw764qh0000gn/T/ipykernel_22908/728900000.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["same_race"] = (df["race_respondent"] == df["race_partner"]).astype("boolean")


race_partner,White,Black,Hispanic,Other,All
race_respondent,,,,,
White,1851,32,91,98,2072
Black,25,164,22,16,227
Hispanic,127,20,180,18,345
Other,89,15,15,85,204
All,2092,231,308,217,2848


## 5. Saving the race data

Save respondent race, partner race and the same-race flag to `data/processed/hcmst2017_race.csv`,
linked to the other files by `caseid_new`. Also save the race distribution of respondents and
partners to `output/tables/`.

In [5]:
# One row per respondent: ID plus the new race columns
race_cols = ["caseid_new", "race_respondent", "race_partner", "same_race"]
df[race_cols].to_csv("../data/processed/hcmst2017_race.csv", index=False)

# Race distribution of respondents and partners
race_dist = pd.DataFrame({
    "respondents": df["race_respondent"].value_counts(),
    "partners": df["race_partner"].value_counts(dropna=False),
}).reindex(order + [None])
race_dist.index = order + ["Missing"]
race_dist = race_dist.fillna(0).astype(int)
race_dist.to_csv("../output/tables/race_distribution.csv")

print("Race file:", df[race_cols].shape)
race_dist

Race file: (2862, 4)


,respondents,partners
White,2082,2092
Black,229,231
Hispanic,347,308
Other,204,217
Missing,0,14


## 6. Final check

Confirm the key numbers. If anything upstream changes, this cell will stop with an error.

In [6]:
assert len(df) == 2862, "Should be 2,862 partnered respondents"
assert df["race_respondent"].isna().sum() == 0, "Every respondent should have a race group"
assert df["race_partner"].isna().sum() == 14, "14 partners should have missing race"
assert (df["race_partner"] == "Hispanic").sum() == 308, "Hispanic origin should be used for partners (308 Hispanic partners)"
assert df["same_race"].sum() == 2280, "Should be 2,280 same-race couples"
assert (df["same_race"] == False).sum() == 568, "Should be 568 interracial couples"

print("All checks passed ✓")

All checks passed ✓
